In [1]:
!pip install transformers torch nltk datasets==2.19.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

# KP20K 로드
dataset = load_dataset("taln-ls2n/kp20k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for taln-ls2n/kp20k contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/taln-ls2n/kp20k
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


BuilderConfig(name='raw', version=0.0.1, data_dir=None, data_files=None, description='This part of my dataset covers the raw data')


Generating train split:   0%|          | 0/530809 [00:00<?, ? examples/s]

In [8]:
sample = dataset["train"].select(range(10))
sample[0]

{'id': 'vXFe8Vy',
 'title': 'virtually enhancing the perception of user actions',
 'abstract': 'This paper proposes using virtual reality to enhance the perception of actions by distant users on a shared application. Here, distance may refer either to space ( e.g. in a remote synchronous collaboration) or time ( e.g. during playback of recorded actions). Our approach consists in immersing the application in a virtual inhabited 3D space and mimicking user actions by animating avatars. We illustrate this approach with two applications, the one for remote collaboration on a shared application and the other to playback recorded sequences of user actions. We suggest this could be a low cost enhancement for telepresence.',
 'keyphrases': ['animation',
  'avatars',
  'telepresence',
  'application sharing',
  'collaborative virtual environments'],
 'prmu': ['P', 'P', 'P', 'R', 'M']}

In [21]:
from transformers import BartForConditionalGeneration, AutoTokenizer

model = BartForConditionalGeneration.from_pretrained("bloomberg/KeyBART")
tokenizer = AutoTokenizer.from_pretrained("bloomberg/KeyBART")

def make_input(sample):
    return sample["title"] + " . " + sample["abstract"]

text = make_input(sample[0])
inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

outputs = model.generate(
    **inputs,
    num_beams=10,
    num_return_sequences=5,
    max_length=40,
    no_repeat_ngram_size=3  # ← 반복 방지
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [22]:
# 디코더
results = tokenizer.batch_decode(outputs, skip_special_tokens=True)
results

['virtual reality;virtual inhabited 3d space;remote synchronous collaboration;telepresence enhancement;user action perception enhancement;remote collaboration;virtual avatars animating avatars;shared application;',
 'virtual reality;virtual inhabited 3d space;remote synchronous collaboration;telepresence enhancement;user action perception enhancement;remote collaboration;virtual avatars animating avatars;distant user',
 'virtual reality;virtual inhabited 3d space;remote synchronous collaboration;telepresence enhancement;user action perception enhancement;remote collaboration;virtual avatars animating avatars;virtual environment;',
 'virtual reality;virtual inhabited 3d space;remote synchronous collaboration;telepresence enhancement;user action perception enhancement;remote collaboration;virtual avatars animating avatars;synchronous',
 'virtual reality;virtual inhabited 3d space;remote synchronous collaboration;telepresence enhancement;user action perception enhancement;remote collabora

In [12]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem(phrase):
    return " ".join([stemmer.stem(w) for w in phrase.split()])

def f1_score(preds, golds, k=5):
    preds_k = preds[:k]
    preds_stemmed = [stem(p) for p in preds_k]
    golds_stemmed = [stem(g) for g in golds]

    match = len(set(preds_stemmed) & set(golds_stemmed))
    precision = match / len(preds_k) if preds_k else 0
    recall = match / len(golds_stemmed) if golds_stemmed else 0

    if precision + recall == 0:
        return 0
    return 2 * precision * recall / (precision + recall)